In [2]:
# =============================================================================
# CHAPTER 2: END-TO-END MACHINE LEARNING PROJECT — Setup & Data Split
# =============================================================================
#
# GOAL: Predict median house value for California districts using census data.
#
# WORKFLOW:
#   Cell 0 → Load data, stratified train/test split, separate features/labels
#   Cell 1 → Full preprocessing pipeline (24 engineered features)
#   Cell 2 → Train LinearRegression baseline, cross-validated RMSE
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import tarfile
import urllib.request

# sklearn — data splitting & evaluation
from sklearn.model_selection import train_test_split, cross_val_score

# sklearn — imputation & encoding
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

# sklearn — pipeline & composition
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector

# sklearn — clustering & similarity (used in ClusterSimilarity)
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics.pairwise import rbf_kernel

# sklearn — models
from sklearn.linear_model import LinearRegression


# --- Load data ---
# Dataset: California Housing (20,640 districts, 1990 census)
# Downloads and extracts from GitHub on first run, then reads from local CSV.

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))


housing_full = load_housing_data()


# --- Stratified train/test split ---
# median_income is the strongest predictor (+0.69 correlation with target).
# A random split risks skewing its distribution by chance.
# Binning into 5 income categories and using stratify= ensures both sets
# are proportionally representative of the full income range.

housing_full["income_cat"] = pd.cut(
    housing_full["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

strat_train_set, strat_test_set = train_test_split(
    housing_full, test_size=0.2, stratify=housing_full["income_cat"], random_state=42
)
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

# Save test set to disk and remove from memory — not touched until final eval.
strat_test_set.to_csv("datasets/housing/test_set.csv", index=False)
del strat_test_set


# --- Separate features and target ---
# housing        → feature matrix (9 columns, includes 1 categorical)
# housing_labels → target vector  (median_house_value)

housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

missing = housing.isnull().sum()
missing = missing[missing > 0]
print(f"Training rows  : {housing.shape[0]:,}")
print(f"Features       : {housing.shape[1]}")
print(f"Missing values : {', '.join(f'{col} ({n})' for col, n in missing.items())}")

Training rows  : 16,512
Features       : 9
Missing values : total_bedrooms (168)


In [3]:
# =============================================================================
# FULL PREPROCESSING PIPELINE
# =============================================================================
#
# Consolidates all transformations arrived at through experimentation.
#
# FEATURE ENGINEERING DECISIONS:
#   - Ratio features (bedrooms, rooms_per_house, people_per_house):
#       Raw counts correlate poorly; ratios reveal density → better predictor.
#   - Log transform (bedrooms, rooms, population, households, income):
#       Long right-tails compressed to roughly Gaussian.
#       housing_median_age already roughly uniform — no log needed.
#   - Cluster similarity (latitude, longitude):
#       10 KMeans centroids capture "Bay Area effect", "LA effect" etc.
#       as smooth continuous features instead of raw coordinates.
#   - One-hot (ocean_proximity):
#       Nominal category, no natural ordering → binary columns.
#   - Remainder (housing_median_age):
#       Imputed + scaled, no other engineering needed.
#
# OUTPUT: (16512, 24)
#   3 ratio + 5 log + 10 cluster + 5 one-hot + 1 remainder
# =============================================================================


# --- ClusterSimilarity (custom sklearn transformer) ---
# Learns k geographic cluster centroids via KMeans, then computes RBF
# (Gaussian) similarity of each district to each centroid.
#   fit()       → KMeans on lat/lon → learns k centroids
#   transform() → rbf_kernel(X, centroids) → similarity matrix
#   K(x, c) = exp(-gamma * ||x - c||²)  →  1.0 at centroid, ~0.0 far away
# Why not raw lat/lon? A linear model can't interpret coordinates as spatial
# proximity. Cluster similarity gives it 10 smooth "nearness" features.

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]


# --- Ratio pipeline helper ---
# FunctionTransformer(column_ratio) wraps a plain function as a sklearn
# transformer so it can live inside a Pipeline/ColumnTransformer.
#   feature_names_out=ratio_name → custom naming function for the output column
# Each ratio pipeline: impute NaN → compute X[:,0]/X[:,1] → standardize

def column_ratio(X):
    return X[:, [0]] / X[:, [1]]

def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler(),
    )


# --- Log pipeline ---
# FunctionTransformer(np.log) applies log element-wise.
#   feature_names_out="one-to-one" → output names match input names
# Impute first (log of NaN would propagate), then log, then standardize.
# Compresses long right-tails → roughly Gaussian → better for linear models.

log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler(),
)


# --- Categorical pipeline ---
# SimpleImputer(strategy="most_frequent") fills any NaN with the mode.
# OneHotEncoder creates one binary column per category (5 for ocean_proximity).
#   Chosen over OrdinalEncoder: ocean_proximity has NO natural ordering —
#   integers would imply false magnitude (ISLAND=2 is not "twice" INLAND=1).
#   handle_unknown="ignore" → unseen categories at inference → all zeros
#   rather than raising an error.

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)


# --- Default numeric pipeline ---
# SimpleImputer(strategy="median") fills NaN with training median.
#   Why median over mean? Robust to outliers (a few $500k-capped values
#   would skew the mean but not the median).
#   Why not drop NaN rows? At inference you can't drop — must handle consistently.
# StandardScaler standardizes to mean=0, std=1 (z-score).
#   Chosen over MinMaxScaler: less sensitive to outliers (outliers shift
#   min/max for ALL rows in MinMaxScaler). Most features roughly Gaussian.
#   Always fit on training data only — apply same params to new data.

default_num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)


# --- Cluster similarity instance ---
# 10 clusters, gamma=1.0 (controls RBF bell-curve width)

cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)


# --- ColumnTransformer: assemble everything ---
# Applies different pipelines to different column subsets in parallel,
# then concatenates all outputs into a single NumPy array.
#   remainder=default_num_pipeline → catches housing_median_age (the only
#   column not explicitly assigned) and imputes + scales it, rather than
#   silently dropping it.
# make_column_selector(dtype_include=object) dynamically selects all
#   object-dtype columns (i.e. categorical). More robust than hardcoding
#   column names if the schema ever changes.

preprocessing = ColumnTransformer([
    ("bedrooms",         ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
    ("rooms_per_house",  ratio_pipeline(), ["total_rooms", "households"]),
    ("people_per_house", ratio_pipeline(), ["population", "households"]),
    ("log",              log_pipeline,     ["total_bedrooms", "total_rooms",
                                            "population", "households", "median_income"]),
    ("geo",              cluster_simil,    ["latitude", "longitude"]),
    ("cat",              cat_pipeline,     make_column_selector(dtype_include=object)),
], remainder=default_num_pipeline)

housing_prepared = preprocessing.fit_transform(housing)

print(f"Output shape : {housing_prepared.shape}")
print(f"\nFeature names:")
for name in preprocessing.get_feature_names_out():
    print(f"  {name}")


# --- Quick reference: pipeline → feature mapping ---

print("\n" + "=" * 60)
print("PIPELINE → FEATURE MAPPING")
print("=" * 60)
print(f"""
  Pipeline              Raw columns                    → Output features
  ─────────────────────────────────────────────────────────────────────
  bedrooms              total_bedrooms, total_rooms    → 1 ratio
  rooms_per_house       total_rooms, households        → 1 ratio
  people_per_house      population, households         → 1 ratio
  log                   bedrooms, rooms, pop, hh, inc  → 5 log-scaled
  geo (ClusterSimilar.) latitude, longitude            → {cluster_simil.n_clusters} cluster similarities
  cat (OneHotEncoder)   ocean_proximity                → 5 one-hot
  remainder             housing_median_age             → 1 scaled
                                                         ──
                                                         {housing_prepared.shape[1]} total
""")

Output shape : (16512, 24)

Feature names:
  bedrooms__ratio
  rooms_per_house__ratio
  people_per_house__ratio
  log__total_bedrooms
  log__total_rooms
  log__population
  log__households
  log__median_income
  geo__Cluster 0 similarity
  geo__Cluster 1 similarity
  geo__Cluster 2 similarity
  geo__Cluster 3 similarity
  geo__Cluster 4 similarity
  geo__Cluster 5 similarity
  geo__Cluster 6 similarity
  geo__Cluster 7 similarity
  geo__Cluster 8 similarity
  geo__Cluster 9 similarity
  cat__ocean_proximity_<1H OCEAN
  cat__ocean_proximity_INLAND
  cat__ocean_proximity_ISLAND
  cat__ocean_proximity_NEAR BAY
  cat__ocean_proximity_NEAR OCEAN
  remainder__housing_median_age

PIPELINE → FEATURE MAPPING

  Pipeline              Raw columns                    → Output features
  ─────────────────────────────────────────────────────────────────────
  bedrooms              total_bedrooms, total_rooms    → 1 ratio
  rooms_per_house       total_rooms, households        → 1 ratio
  people_per_ho

In [4]:
# =============================================================================
# TRAIN & EVALUATE
# =============================================================================
#
# Establishes a performance baseline before trying complex models.
# Cross-validated RMSE is the honest metric — training RMSE alone is misleading
# because the model has already seen the training data.
# =============================================================================


# --- Linear Regression baseline ---
# sklearn.linear_model.LinearRegression
#   Fits a hyperplane by minimizing residual sum of squares (OLS).
#   No hyperparameters to tune — the simplest possible baseline.
#   If a complex model can't beat this, something is wrong upstream.
#
# make_pipeline(preprocessing, LinearRegression())
#   Wraps preprocessing + model together. During cross-validation,
#   preprocessing.fit() runs ONLY on training folds — no data leakage.

lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_reg.fit(housing, housing_labels)


# --- Cross-validated RMSE (10-fold) ---
# cross_val_score splits the training set into 10 folds:
#   Trains on 9, evaluates on 1, rotates through all 10.
#   Averages the 10 scores → stable estimate of generalization error.
#
# scoring="neg_root_mean_squared_error":
#   sklearn convention: higher = better, so RMSE is negated. We negate back.
#
# RMSE is in the same units as the target (dollars).
#   Penalizes large errors more than MAE (squares before rooting).
#   This baseline is the floor to beat with RandomForest, SVR, etc.

rmse_scores = -cross_val_score(
    lin_reg, housing, housing_labels,
    scoring="neg_root_mean_squared_error", cv=10
)

print(f"CV RMSE scores : {rmse_scores.round(0)}")
print(f"Mean RMSE      : ${rmse_scores.mean():,.0f}")
print(f"Std RMSE       : ${rmse_scores.std():,.0f}")


# --- Full statistical summary ---
# describe() reveals the story the mean hides: min/max spread, percentiles,
# and whether any folds are outliers. Useful for spotting instability.

print("\nCV RMSE distribution:")
pd.Series(rmse_scores).describe()


# --- Sample predictions vs actuals ---

sample = housing.iloc[:5]
predictions = lin_reg.predict(sample)

print("\nSample predictions vs actuals:")
pd.DataFrame({
    "actual":    housing_labels.iloc[:5].values,
    "predicted": predictions.round(0),
    "error":     (predictions - housing_labels.iloc[:5].values).round(0),
})

CV RMSE scores : [69857. 68417. 65505. 81038. 69063. 68425. 67952. 71179. 68091. 70508.]
Mean RMSE      : $70,003
Std RMSE       : $3,968

CV RMSE distribution:

Sample predictions vs actuals:


,actual,predicted,error
0,458300.0,245970.0,-212330.0
1,483800.0,372738.0,-111062.0
2,101700.0,135707.0,34007.0
3,96100.0,91441.0,-4659.0
4,361800.0,330874.0,-30926.0


In [5]:
# =============================================================================
# DECISION TREE REGRESSOR
# =============================================================================
#
# Decision Trees split data into regions by learning threshold rules on
# features (e.g., "if median_income < 3.5 and cluster_2_similarity > 0.7").
# Unlike LinearRegression, they can capture non-linear relationships and
# feature interactions without explicit feature engineering.
#
# Key question: Can a non-linear model beat the $70k RMSE baseline?
# =============================================================================

from sklearn.tree import DecisionTreeRegressor

tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))
tree_reg.fit(housing, housing_labels)


# --- Cross-validated RMSE (10-fold) ---
# Training RMSE for a Decision Tree is typically ~0 (perfect memorization),
# which is why CV is essential — it reveals overfitting.

rmse_scores = -cross_val_score(
    tree_reg, housing, housing_labels,
    scoring="neg_root_mean_squared_error", cv=10
)

print(f"CV RMSE scores : {rmse_scores.round(0)}")
print(f"Mean RMSE      : ${rmse_scores.mean():,.0f}")
print(f"Std RMSE       : ${rmse_scores.std():,.0f}")


# --- Full statistical summary ---
# describe() reveals the story the mean hides: min/max spread, percentiles,
# and whether any folds are outliers. Compare with LinearRegression above.

print("\nCV RMSE distribution:")
pd.Series(rmse_scores).describe()


# --- Sample predictions vs actuals ---

sample = housing.iloc[:5]
predictions = tree_reg.predict(sample)

print("\nSample predictions vs actuals:")
pd.DataFrame({
    "actual":    housing_labels.iloc[:5].values,
    "predicted": predictions.round(0),
    "error":     (predictions - housing_labels.iloc[:5].values).round(0),
})

CV RMSE scores : [64608. 66409. 66203. 65864. 68087. 66535. 66923. 68532. 66367. 66209.]
Mean RMSE      : $66,574
Std RMSE       : $1,047

CV RMSE distribution:

Sample predictions vs actuals:


,actual,predicted,error
0,458300.0,458300.0,0.0
1,483800.0,483800.0,0.0
2,101700.0,101700.0,0.0
3,96100.0,96100.0,0.0
4,361800.0,361800.0,0.0


In [6]:
# =============================================================================
# RANDOM FOREST REGRESSOR
# =============================================================================
#
# A Random Forest is an ensemble of Decision Trees. It fixes the overfitting
# problem by training many trees on random subsets of data AND features,
# then averaging their predictions. The individual trees still memorize,
# but their random errors cancel out when averaged → lower variance.
#
# Key question: Can an ensemble beat both the linear ($70k) and tree ($66k)
# baselines while avoiding overfitting?
# =============================================================================

from sklearn.ensemble import RandomForestRegressor

forest_reg = make_pipeline(preprocessing, RandomForestRegressor(
    n_estimators=100, random_state=42
))
forest_reg.fit(housing, housing_labels)


# --- Cross-validated RMSE (10-fold) ---

rmse_scores = -cross_val_score(
    forest_reg, housing, housing_labels,
    scoring="neg_root_mean_squared_error", cv=10
)

print(f"CV RMSE scores : {rmse_scores.round(0)}")
print(f"Mean RMSE      : ${rmse_scores.mean():,.0f}")
print(f"Std RMSE       : ${rmse_scores.std():,.0f}")


# --- Full statistical summary ---

print("\nCV RMSE distribution:")
pd.Series(rmse_scores).describe()


# --- Sample predictions vs actuals ---

sample = housing.iloc[:5]
predictions = forest_reg.predict(sample)

print("\nSample predictions vs actuals:")
pd.DataFrame({
    "actual":    housing_labels.iloc[:5].values,
    "predicted": predictions.round(0),
    "error":     (predictions - housing_labels.iloc[:5].values).round(0),
})

CV RMSE scores : [46577. 47239. 45496. 46488. 45911. 47731. 47561. 49141. 47121. 47116.]
Mean RMSE      : $47,038
Std RMSE       : $969

CV RMSE distribution:

Sample predictions vs actuals:


,actual,predicted,error
0,458300.0,434707.0,-23593.0
1,483800.0,477010.0,-6790.0
2,101700.0,110917.0,9217.0
3,96100.0,99881.0,3781.0
4,361800.0,367395.0,5595.0


In [7]:
# =============================================================================
# GRID SEARCH — HYPERPARAMETER TUNING (Random Forest)
# =============================================================================
#
# Our Random Forest used all defaults and got $47k RMSE. Can we do better?
#
# This grid tunes BOTH the preprocessing and the model together:
#   - preprocessing__geo__n_clusters: how many geographic clusters
#     (changes the feature count — 5 clusters = 5 similarity features, 15 = 15)
#   - random_forest__max_features: how many features each tree can consider
#     at each split (lower = more diversity between trees)
#
# Using Pipeline() with named steps instead of make_pipeline() so that
# parameter paths are readable:
#   "preprocessing__geo__n_clusters"   → preprocessing → geo → n_clusters
#   "random_forest__max_features"      → random_forest → max_features
#
# param_grid is a LIST of two dicts — GridSearchCV tries all combinations
# from each dict separately:
#   Grid 1: 3 × 3 = 9 combos  (fewer clusters × fewer features per split)
#   Grid 2: 2 × 3 = 6 combos  (more clusters × more features per split)
#   Total: 15 combinations × 3 folds = 45 training runs
# =============================================================================

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestRegressor(random_state=42)),
])

param_grid = [
    {'preprocessing__geo__n_clusters': [5, 8, 10],
     'random_forest__max_features': [4, 6, 8]},
    {'preprocessing__geo__n_clusters': [10, 15],
     'random_forest__max_features': [6, 8, 10]},
]

grid_search = GridSearchCV(
    full_pipeline,
    param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
)

grid_search.fit(housing, housing_labels)


# --- Best hyperparameters and score ---

print("Best hyperparameters:")
for param, value in grid_search.best_params_.items():
    name = param.split("__")[-1]
    print(f"  {name}: {value}")

best_rmse = -grid_search.best_score_
print(f"\nBest CV RMSE: ${best_rmse:,.0f}")
print(f"(vs default Random Forest: $47,038)")


# --- All results ranked by CV RMSE ---

results = pd.DataFrame(grid_search.cv_results_)
results["mean_rmse"] = -results["mean_test_score"]
results["std_rmse"] = results["std_test_score"]

display = results[[
    "param_preprocessing__geo__n_clusters",
    "param_random_forest__max_features",
    "mean_rmse", "std_rmse",
]].sort_values("mean_rmse")

display.columns = ["n_clusters", "max_features", "CV RMSE", "Std RMSE"]

print("\nAll combinations ranked by CV RMSE:")
display.round(0)

Best hyperparameters:
  n_clusters: 15
  max_features: 6

Best CV RMSE: $43,590
(vs default Random Forest: $47,038)

All combinations ranked by CV RMSE:


,n_clusters,max_features,CV RMSE,Std RMSE
12,15,6,43590.0,663.0
13,15,8,44066.0,604.0
7,10,6,44278.0,520.0
9,10,6,44278.0,520.0
6,10,4,44279.0,495.0
14,15,10,44682.0,378.0
3,8,4,44748.0,330.0
8,10,8,44882.0,313.0
10,10,8,44882.0,313.0
4,8,6,45111.0,384.0
